## 1. Environment Setup & Data Loading
Initialization of the environment and loading the cleaned dataset. Working with a `.parquet` file ensures faster read/write speeds and smaller file sizes compared to standard CSV files.

In [17]:
# Import required data manipulation libraries
import pandas as pd
import numpy as np

In [18]:
# Load the cleaned dataset using the PyArrow engine for optimal performance
df = pd.read_parquet("df_cleaned.parquet", engine= "pyarrow")

## 2. Feature Engineering
This section creates new foundational metrics to analyze shipping efficiency, profitability, and sales performance tiers.

**Questions Answered:**
* **`Shipping_Duration`:** "How many days does it take to process and ship an order?"
* **`Profit_Margin`:** "What percentage of the sales revenue is retained as net profit?"
* **`Sales Performance Category`:** "How do individual transactions classify into distinct volume tiers (Low, Medium, High, Premium)?"

In [19]:
def engineer_sales_features(df) :
    # 1. Work on a copy to prevent unintended modifications to the original DataFrame
    data = df.copy()
    
    # 2. Calculate Shipping Duration (in days) by subtracting Order Date from Ship Date
    data["Shipping_Duration"] = (data["Ship Date"] - data["Order Date"]).dt.days
    
    # 3. Calculate Profit Margin Percentage (safely handling zero division)
    data["Profit_Margin"] = (data["Profit"] / data["Sales"])*100 
        
    # 4. Bin Sales into categorical Performance Tiers for easier grouping
    data["Sales Performance Category"] = pd.cut(
        data["Sales"],
        bins=[data["Sales"].min() - 1, 20, 100, 500, data["Sales"].max() + 1],
        labels=["Low Sales", "Medium Sales", "High Sales", "Premium Sales"]
    )
    
    return data

df = engineer_sales_features(df)

#### Discount & Profitability Analysis

**`Question:`** "Which customer segments and product sub-categories are 'Discount Addicts' that drive high sales volume but systematically destroy net profit?"

In [20]:
def analyze_discount(df):

    # Group data by Category and Sub-Category to aggregate key financial metrics
    discount_analysis = df.groupby(["Category", "Sub-Category"], observed=True).agg(
            Total_Sales=("Sales", "sum"),
            Total_Profit=("Profit", "sum"),
            Total_Orders=("Order ID", "count"),
            Avg_Discount=("Discount", "mean"),
            Min_Discount=("Discount", "min"),
            Max_Discount=("Discount", "max"),
        ).reset_index()

    # Calculate the overall Profit Margin for each Sub-Category
    discount_analysis["Profit_Margin"] = (discount_analysis["Total_Profit"] / discount_analysis["Total_Sales"]) * 100

    return discount_analysis

analyze_discount(df)
# --- FINDINGS ---
#  Loss-Making Sub-Categories: Three sub-categories are actively destroying net profit despite generating sales.
#   - Furniture - Bookcases: Generated $114,879 in sales but resulted in a -$3,472 net loss (Profit margin: -3.02%).
#   - Furniture - Tables: Generated $206,965 in sales but resulted in a -$17,725 net loss (Profit margin: -8.56%).
#   - Office Supplies - Supplies: Generated $46,673 in sales but resulted in a -$1,189 net loss (Profit margin: -2.54%).
#  Highest Margins: Office Supplies sub-categories like Labels (44.41%), Paper (43.39%), and Envelopes (42.26%) drive the highest relative profit margins.


,Category,Sub-Category,Total_Sales,Total_Profit,Total_Orders,Avg_Discount,Min_Discount,Max_Discount,Profit_Margin
0,Furniture,Bookcases,114879.9963,-3472.5560,228,0.211140,0.0,0.7,-3.022768
1,Furniture,Chairs,328449.1030,26590.1663,617,0.170178,0.0,0.3,8.095673
2,Furniture,Furnishings,91705.1640,13059.1436,957,0.138349,0.0,0.6,14.240358
3,Furniture,Tables,206965.5320,-17725.4811,319,0.261285,0.0,0.5,-8.564460
4,Office Supplies,Appliances,107532.1610,18138.0054,466,0.166524,0.0,0.8,16.867517
5,Office Supplies,Art,27118.7920,6527.7870,796,0.074874,0.0,0.2,24.071083
6,Office Supplies,Binders,203412.7330,30221.7633,1523,0.372292,0.0,0.8,14.857361
7,Office Supplies,Envelopes,16476.4020,6964.1767,254,0.080315,0.0,0.2,42.267582
8,Office Supplies,Fasteners,3024.2800,949.5182,217,0.082028,0.0,0.2,31.396504
9,Office Supplies,Labels,12486.3120,5546.2540,364,0.068681,0.0,0.2,44.418672


#### Customer Retention Risk (The "One-Time Buyer" Leak)
**`Question:`** "What percentage of our customers buy once and never come back?"

**`Why it’s critical:`** Acquiring a new customer costs 5x more than retaining an existing one. If specific products lead to high one-time purchases without repeat sales, those products might have quality issues or poor customer satisfaction.

In [21]:
def calculate_one_time_buyer_pct(df):
    # 1. Count unique orders per customer
    order_counts = df.groupby("Customer ID")["Order ID"].nunique()
    
    # 2. Calculate percentage directly using boolean mean
    one_time_pct = (order_counts == 1).mean() * 100
    
    return round(one_time_pct, 2)

pct = calculate_one_time_buyer_pct(df)
print(f"One-Time Buyer Rate: {pct}%")

# --- FINDINGS ---
#  The store has an exceptionally low one-time buyer rate of exactly 1.51%. 
#  This indicates incredibly strong customer retention and loyalty.

One-Time Buyer Rate: 1.51%


#### Order Size Profitability
**`Question:`** "Is processing smaller 'Micro' orders financially viable compared to larger enterprise-level orders?"

In [22]:
def analyze_order_size_profitability(df):
    # 2. Aggregate line items to Order-level totals
    order_level = df.groupby('Order ID').agg(
        Order_Sales=('Sales', 'sum'),
        Order_Profit=('Profit', 'sum')
    ).reset_index()

    # 3. Bin total order values into size tiers
    order_level['Order_Size_Tier'] = pd.cut(
        order_level['Order_Sales'],
        bins=[0, 20, 100, 500, np.inf],
        labels=['Micro (<$20)', 'Small ($20-$100)', 'Medium ($100-$500)', 'Large (>$500)']
    )

    # 4. Group by tier and compute summary metrics
    summary = order_level.groupby('Order_Size_Tier', observed=False).agg(
        Total_Orders=('Order ID', 'count'),
        Total_Sales=('Order_Sales', 'sum'),
        Total_Profit=('Order_Profit', 'sum'),
        Avg_Profit_Per_Order=('Order_Profit', 'mean')
    ).reset_index()

    # 4. Calculate the overarching profit margin per tier
    summary["Profit_Margin"] = (summary["Total_Profit"] / summary["Total_Sales"]) * 100

    return summary
analyze_order_size_profitability(df)

# --- FINDINGS ---
# Volume vs. Value: Large orders (>$500) make up the bulk of net profit, generating $231,193 from just 1,274 orders.
# Micro Orders: Orders under $20 generate only $1.47 of profit per order on average, meaning high operational effort for very low financial reward.

,Order_Size_Tier,Total_Orders,Total_Sales,Total_Profit,Avg_Profit_Per_Order,Profit_Margin
0,Micro (<$20),810,8.774961e+03,1195.6747,1.476142,13.625983
1,Small ($20-$100),1296,6.797159e+04,10414.1959,8.035645,15.321395
2,Medium ($100-$500),1629,4.123148e+05,43593.5748,26.760942,10.572886
3,Large (>$500),1274,1.808139e+06,231193.5763,181.470625,12.786269


#### Revenue Concentration (The 80/20 Rule)
**`Question:`** "Does 80% of our overall net profit depend on the top 20% of our customer base?"

**`Why it’s critical:`** Identifies revenue concentration risk. If a tiny fraction of customers drives all your profits, losing just a few of them could cripple the business.

In [23]:
def analyze_revenue_concentration(df):
    data = df.copy()
    
    # 1. Aggregate total net profit per customer
    cust_profit = data.groupby('Customer ID')['Profit'].sum().reset_index()

    # 2. Sort customers from highest profit to lowest
    cust_profit = cust_profit.sort_values(by='Profit', ascending=False).reset_index(drop=True)

    # 3. Calculate the total global net profit and identify the top 20% threshold
    total_net_profit = cust_profit['Profit'].sum()
    top_20_count = int(np.ceil(0.20 * len(cust_profit)))
    
    # 4. Sum the profit of the top 20% and determine their percentage share
    top_20_profit = cust_profit.iloc[:top_20_count]['Profit'].sum()
    top_20_share = (top_20_profit / total_net_profit) * 100
    
    return round(top_20_share, 2)
analyze_revenue_concentration(df)

# --- FINDINGS ---
#  The business adheres closely to the Pareto Principle; the top 20% of the customer base accounts for 81.66% of the company's total net profit.

np.float64(81.66)

#### Time Series & Trend Analysis
**`Question:`** "What are the seasonal trends and anomalies in profitability over time?"

Aggregating data monthly to detect seasonal trends and potential anomalies in profitability.

In [24]:
def time_series(df):
    
    # 1. Create a Year-Month column
    df["Order Month"] = df["Order Date"].dt.to_period("M")

    # 2. Aggregate sales and profit 
    df_time_series = df.groupby("Order Month").agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum")
    ).reset_index()
    
    # 3. Calculate monthly Profit Margin
    df_time_series["Profit_Margin"] = (df_time_series["Profit"] / df_time_series["Sales"]) * 100

    # 4. Convert period back to timestamp 
    df_time_series["Order Month"] = df_time_series["Order Month"].dt.to_timestamp()

    return df_time_series

time_series(df).head(10)

# --- FINDINGS ---
# * Profitability fluctuates heavily month-to-month. For example, July 2016 experienced a loss with a -2.47% margin, whereas February 2016 achieved an excellent 19.07% margin.
# * January 2017 was the worst recorded month, operating at a severe -18.05% margin.

,Order Month,Sales,Profit,Profit_Margin
0,2016-01-01,14236.8950,2450.1907,17.210148
1,2016-02-01,4519.8920,862.3084,19.078075
2,2016-03-01,55691.0090,498.7299,0.895530
3,2016-04-01,28295.3450,3488.8352,12.330068
4,2016-05-01,23648.2870,2738.7096,11.581006
5,2016-06-01,34595.1276,4976.5244,14.385044
6,2016-07-01,33946.3930,-841.4826,-2.478857
7,2016-08-01,27909.4685,5318.1050,19.054842
8,2016-09-01,81777.3508,8328.0994,10.183870
9,2016-10-01,31453.3930,3448.2573,10.963069


## 2. Data Validation & Quality Checks
Ensuring there are no infinite or missing values in our features before proceeding to deeper analysis.

In [25]:
def validate_features(df):
    # 1. Check numeric columns for missing or infinite values
    cols = df.select_dtypes(include= ["number"]).columns.tolist()
    print("Validate Numerical Values: ")
    print("Infinite values:\n", np.isinf(df[cols]).sum())
    print("Null Values:\n", df[cols].isna().sum())

    # 2. Check categorical columns for missing values
    cols = df.select_dtypes(include= ["category"]).columns.tolist()
    print("Validate cateforical Values: ")
    print("Null Values:\n", df[cols].isna().sum())

# Execute validation across the primary and aggregated datasets
validate_features(df)
validate_features(analyze_discount(df))
validate_features(analyze_order_size_profitability(df))
validate_features(time_series(df))

# --- FINDINGS ---
# * The dataset is perfectly clean. There are zero null values and zero infinite values across all numeric and categorical columns.


Validate Numerical Values: 
Infinite values:
 Sales                0
Quantity             0
Discount             0
Profit               0
Shipping_Duration    0
Profit_Margin        0
dtype: int64
Null Values:
 Sales                0
Quantity             0
Discount             0
Profit               0
Shipping_Duration    0
Profit_Margin        0
dtype: int64
Validate cateforical Values: 
Null Values:
 Ship Mode                     0
Segment                       0
Country/Region                0
State                         0
Region                        0
Category                      0
Sub-Category                  0
Sales Performance Category    0
dtype: int64
Validate Numerical Values: 
Infinite values:
 Total_Sales      0
Total_Profit     0
Total_Orders     0
Avg_Discount     0
Min_Discount     0
Max_Discount     0
Profit_Margin    0
dtype: int64
Null Values:
 Total_Sales      0
Total_Profit     0
Total_Orders     0
Avg_Discount     0
Min_Discount     0
Max_Discount     0
Profi

## 3. Exploratory Data Analysis (EDA)
A deep dive into high-level statistics, distributions, and geographical performance.

In [26]:
def describtive_statistics(df):

    df_clean = df.copy()
    # Output descriptive stats for numerical, categorical, and string types
    display(df_clean.describe())
    display(df_clean.describe(include="category"))
    display(df_clean.describe(include="string"))
    
describtive_statistics(df)

,Order Date,Ship Date,Sales,Quantity,Discount,Profit,Shipping_Duration,Profit_Margin
count,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,2018-04-30 10:03:51.979187712,2018-05-04 09:03:29.645787392,229.858001,3.789574,0.156203,28.656896,3.958075,12.031393
min,2016-01-03 00:00:00,2016-01-07 00:00:00,0.444000,1.000000,0.000000,-6599.978000,0.000000,-275.000000
25%,2017-05-23 00:00:00,2017-05-27 00:00:00,17.280000,2.000000,0.000000,1.728750,3.000000,7.500000
50%,2018-06-26 00:00:00,2018-06-29 00:00:00,54.490000,3.000000,0.200000,8.666500,4.000000,27.000000
75%,2019-05-14 00:00:00,2019-05-18 00:00:00,209.940000,5.000000,0.200000,29.364000,5.000000,36.250000
max,2019-12-30 00:00:00,2020-01-05 00:00:00,22638.480000,14.000000,0.800000,8399.976000,7.000000,50.000000
std,NaN,NaN,623.245101,2.225110,0.206452,234.260108,1.747937,46.675435


,Ship Mode,Segment,Country/Region,State,Region,Category,Sub-Category,Sales Performance Category
count,9994,9994,9994,9994,9994,9994,9994,9994
unique,4,3,1,49,4,3,17,4
top,Standard Class,Consumer,United States,California,West,Office Supplies,Binders,Medium Sales
freq,5968,5191,9994,2001,3203,6026,1523,3376


,Order ID,Customer ID,Customer Name,City,Postal Code,Product ID,Product Name
count,9994,9994,9994,9994,9994,9994,9994
unique,5009,793,793,531,631,1862,1850
top,Ca-2019-100111,Wb-21850,William Brown,New York City,10035.0,Off-Pa-10001970,Staple Envelope
freq,14,37,37,915,263,19,48


In [27]:
# View the distribution of sales performance tiers
print(df["Sales Performance Category"].value_counts())

# Group orders by their delivery timeframe
print(df.groupby("Shipping_Duration").agg(Orders=("Shipping_Duration", "count")))

Sales Performance Category
Medium Sales     3376
Low Sales        2853
High Sales       2603
Premium Sales    1162
Name: count, dtype: int64
                   Orders
Shipping_Duration        
0                     519
1                     369
2                    1336
3                    1004
4                    2772
5                    2170
6                    1202
7                     622


In [28]:
# Aggregate Sales, Profit, and Discount by State
def display_worst_performing_states(df):
    
    
    state_profit = df.groupby("State", observed=True)[["Sales", "Profit", "Discount"]].mean().reset_index()

    # Calculate Profit Margin as a percentage 
    state_profit["Margin"] = (state_profit["Profit"] / state_profit["Sales"]) * 100

    # Sort by Margin in ascending order to find the lowest 
    worst_states = state_profit.sort_values("Margin", ascending=True).head(10)
    
    # Display the results
    print("Top 10 Worst-Performing States by Profit Margin:")
    print(worst_states)
    
    return worst_states

# Execute the function
display_worst_performing_states(df)

# --- FINDINGS ---
# Geographical Losses: Certain states are actively losing the business money. Ohio is the worst performer (-21.68% margin), followed closely by Colorado (-20.33%) and Tennessee (-17.42%).

Top 10 Worst-Performing States by Profit Margin:
             State       Sales     Profit  Discount     Margin
33            Ohio  166.861697 -36.186304  0.324947 -21.686405
4         Colorado  176.418231 -35.867351  0.316484 -20.330864
40       Tennessee  167.551219 -29.189583  0.291257 -17.421289
11        Illinois  162.939230 -25.625787  0.390041 -15.727205
41           Texas  172.779742 -26.121174  0.370193 -15.118192
31  North Carolina  223.305880 -30.083985  0.283534 -13.472097
36    Pennsylvania  198.487077 -26.507598  0.328620 -13.354823
1          Arizona  157.508933 -15.303235  0.303571  -9.715789
35          Oregon  140.573790  -9.600569  0.288710  -6.829558
8          Florida  233.612815  -8.875461  0.299347  -3.799219


,State,Sales,Profit,Discount,Margin
33,Ohio,166.861697,-36.186304,0.324947,-21.686405
4,Colorado,176.418231,-35.867351,0.316484,-20.330864
40,Tennessee,167.551219,-29.189583,0.291257,-17.421289
11,Illinois,162.939230,-25.625787,0.390041,-15.727205
41,Texas,172.779742,-26.121174,0.370193,-15.118192
31,North Carolina,223.305880,-30.083985,0.283534,-13.472097
36,Pennsylvania,198.487077,-26.507598,0.328620,-13.354823
1,Arizona,157.508933,-15.303235,0.303571,-9.715789
35,Oregon,140.573790,-9.600569,0.288710,-6.829558
8,Florida,233.612815,-8.875461,0.299347,-3.799219


In [29]:
# Review the worst months
time_series(df).sort_values("Profit_Margin", ascending = True).head(10)

,Order Month,Sales,Profit,Profit_Margin
12,2017-01-01,18174.0756,-3281.0070,-18.053226
6,2016-07-01,33946.3930,-841.4826,-2.478857
2,2016-03-01,55691.0090,498.7299,0.895530
39,2019-04-01,36521.5361,933.2900,2.555451
34,2018-11-01,79411.9658,4011.4075,5.051389
31,2018-08-01,31115.3743,2062.0693,6.627172
26,2018-03-01,51715.8750,3611.9680,6.984254
27,2018-04-01,38750.0390,2977.8149,7.684676
37,2019-02-01,20301.1334,1613.8720,7.949665
46,2019-11-01,118447.8250,9690.1037,8.180905


In [30]:
# Identify individual customers who are bleeding company profits
def display_loss_making_customers(df):
    # Group by Customer ID and calculate total profit per customer
    cust_summary = df.groupby("Customer ID")["Profit"].sum().reset_index()
    
    # Filter for customers with a net profit less than 0
    loss_cust = cust_summary[cust_summary["Profit"] < 0]
    
    # Sort and display the top 5 biggest loss-makers
    print("Top 5 Loss-Making Customers:")
    print(loss_cust.sort_values("Profit", ascending=True).head(5))
    
    print(f"Number of loss-making customers: {len(loss_cust)}")

# Execute the function
display_loss_making_customers(df)
# --- FINDINGS ---
#  Customer Losses: Out of the customer base, exactly 310 customers have a net-negative lifetime profitability. The single worst customer account (Cs-12505) has cost the business over $6,626.

Top 5 Loss-Making Customers:
    Customer ID     Profit
181    Cs-12505 -6626.3895
310    Gt-14635 -4108.6589
459    Lf-17185 -3583.9770
711    Sr-20425 -3333.9144
322    Hg-14965 -2797.9635
Number of loss-making customers: 155


In [31]:
# Determine if faster/slower shipping correlates with better or worse margins
def display_shipping_duration_profitability(df):
    
    # Group by shipping duration and aggregate the sum of Sales and Profit
    shipping_day_analysis = df.groupby("Shipping_Duration").agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum")
    )
    # Calculate the profit margin as a percentage for each shipping duration
    shipping_day_analysis["shipping_day_margin"] = (shipping_day_analysis["Profit"] / shipping_day_analysis["Sales"]) * 100
    
    # Display the resulting dataframe
    print("Profitability by Shipping Duration (Days):")
    print(shipping_day_analysis)
    
    return shipping_day_analysis

# Execute the function
display_shipping_duration_profitability(df)

# --- FINDINGS ---
# Shipping Consistency: Most orders ship within 4 to 5 days. Shipping duration does not appear to negatively impact profit margins, with margins remaining stable between 11.09% and 14.42% regardless of delivery speed.

Profitability by Shipping Duration (Days):
                         Sales      Profit  shipping_day_margin
Shipping_Duration                                              
0                  124907.6910  15385.9685            12.317871
1                   67975.3312   7541.2269            11.094064
2                  368693.5300  53196.0795            14.428265
3                  204448.0908  26791.0719            13.104095
4                  631811.3573  71138.2510            11.259413
5                  494376.5397  58736.6093            11.880946
6                  240271.3778  33275.4184            13.849098
7                  164716.9425  20332.3962            12.343840


,Sales,Profit,shipping_day_margin
Shipping_Duration,,,
0,124907.6910,15385.9685,12.317871
1,67975.3312,7541.2269,11.094064
2,368693.5300,53196.0795,14.428265
3,204448.0908,26791.0719,13.104095
4,631811.3573,71138.2510,11.259413
5,494376.5397,58736.6093,11.880946
6,240271.3778,33275.4184,13.849098
7,164716.9425,20332.3962,12.343840


## 4 Data Export
Exporting the heavily engineered dataset to a Parquet file for dashboarding, machine learning, or future downstream use.

In [32]:
def Save_as_parquet(df,Name):    
    df.to_parquet(f"{Name}.parquet", engine="pyarrow")

Save_as_parquet(df, "df_Feature")